<a href="https://colab.research.google.com/github/cwyforjupas-del/tiktok-stuff/blob/cwyforjupas-del-patch-1/tiktok_stuff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
import json
import os
from datetime import datetime

# ================= CONFIG =================
TEMPLATE_PATH = "Tiktoksellercenter_batchupload_20260617_template.xlsx"
OUTPUT_DIR = "output"
BRAND_NAME = "MUET"
BRAND_ID = "7618173140417627905"
CURRENCY = "SGD"
BATCH_SIZE = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRACK_FILE = "uploaded_products.json"
if os.path.exists(TRACK_FILE):
    with open(TRACK_FILE, 'r') as f:
        uploaded = json.load(f)
else:
    uploaded = []

def save_uploaded():
    with open(TRACK_FILE, 'w') as f:
        json.dump(uploaded, f, indent=2)

def fill_template(selected_products, products_db_to_use):
    try:
        wb = load_workbook(TEMPLATE_PATH)
        ws = wb["Template"]
    except FileNotFoundError:
        print(f"Error: {TEMPLATE_PATH} not found.")
        return None

    start_row = 6
    while ws.cell(row=start_row, column=3).value is not None:
        start_row += 1

    for idx, prod_name in enumerate(selected_products):
        prod = products_db_to_use[prod_name]
        row = start_row + idx

        ws.cell(row=row, column=1, value=prod["category"])
        ws.cell(row=row, column=2, value=BRAND_NAME)
        ws.cell(row=row, column=3, value=prod_name)
        ws.cell(row=row, column=4, value=prod["description"])
        ws.cell(row=row, column=5, value=prod["main_image"])
        ws.cell(row=row, column=14, value="Color")
        ws.cell(row=row, column=15, value=prod["color"])
        ws.cell(row=row, column=19, value=prod.get("weight", 300))
        dims = prod.get("dimensions", (25, 20, 10))
        ws.cell(row=row, column=20, value=dims[0])
        ws.cell(row=row, column=21, value=dims[1])
        ws.cell(row=row, column=22, value=dims[2])
        ws.cell(row=row, column=24, value=prod.get("price", 89.9))
        ws.cell(row=row, column=26, value=100)

        sku = f"MUET-{datetime.now().strftime('%Y%m%d')}-{idx+1:03d}"
        ws.cell(row=row, column=27, value=sku)
        ws.cell(row=row, column=28, value="Yes")

        if prod_name not in uploaded: uploaded.append(prod_name)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    output_path = f"{OUTPUT_DIR}/TikTok_MUET_Batch_{timestamp}.xlsx"
    wb.save(output_path)
    save_uploaded()
    return output_path

if __name__ == "__main__":
    # Ensure products_db is defined locally if not in global scope
    current_db = globals().get('products_db', {})

    if not current_db:
        print("products_db not found. Rebuilding from CSV...")
        if os.path.exists('bauhaus_muet_products.csv'):
             df_temp = pd.read_csv('bauhaus_muet_products.csv')
             current_db = {row['Product Name']: {"category": "Uncategorized", "description": row['Product Name'], "main_image": row['Image URL'], "price": row['Price'], "color": "Multi", "dimensions": (25,20,10)} for _, row in df_temp.iterrows()}
             # Also update global so other cells see it
             globals()['products_db'] = current_db

    if current_db:
        available = [p for p in current_db.keys() if p not in uploaded]
        if not available:
            print("All products have been processed!")
        else:
            selected = available[:BATCH_SIZE]
            print(f"Processing batch of {len(selected)} products...")
            output = fill_template(selected, current_db)
            if output:
                print(f"\n✓ Success! Generated: {output}")
    else:
        print("Error: No product data found. Please run the integration cell (14ca542a) first.")

products_db not found. Rebuilding from CSV...
Error: No product data found. Please run the integration cell (14ca542a) first.


In [ ]:
### Redundant Cell Removed
*Logic consolidated into the 'Integrating Scraped Data' section below.*

FileNotFoundError: [Errno 2] No such file or directory: 'bauhaus_muet_products.csv'

## Web Scraping for MUET Products

This section will contain code to scrape product information from the provided Bauhaus Hong Kong website for MUET collections. We will use `requests` to fetch the webpage and `BeautifulSoup` to parse the HTML.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Define the URL to scrape
URL = "https://www.bauhaus.com.hk/en/collections/muet"

# Send a GET request to the URL
response = requests.get(URL)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')

    products = []

    # --- Assuming product structure based on common e-commerce sites ---
    # This part might need adjustment based on the actual website's HTML structure.
    # Common classes for product items could be 'product-item', 'product-card', 'grid-item', etc.
    # You might need to inspect the page source in your browser to find the correct selectors.

    # Example selectors (these are guesses and might need to be changed):
    # Find all product containers
    product_containers = soup.find_all('div', class_='product-item-info') # Common class for product info

    if not product_containers:
        # Fallback to a broader search if the above class is not found
        product_containers = soup.find_all('li', class_=lambda x: x and ('product-item' in x or 'product-card' in x)) # More general search

    for container in product_containers:
        name = container.find('a', class_='product-item-link') # Common class for product name link
        price = container.find('span', class_='price') # Common class for price
        image = container.find('img', class_='product-image-photo') # Common class for product image

        product_name = name.get_text(strip=True) if name else 'N/A'
        product_price = price.get_text(strip=True) if price else 'N/A'
        product_image_url = image['src'] if image and 'src' in image.attrs else 'N/A'

        products.append({
            'Product Name': product_name,
            'Price': product_price,
            'Image URL': product_image_url
        })

    # Create a Pandas DataFrame from the scraped data
    df_scraped_products = pd.DataFrame(products)

    # Display the first 5 rows of the DataFrame
    print(f"Found {len(df_scraped_products)} products.")
    display(df_scraped_products.head())

else:
    print(f"Failed to retrieve the web page. Status code: {response.status_code}")
    df_scraped_products = pd.DataFrame()


Found 16 products.


,Product Name,Price,Image URL
0,N/A,N/A,N/A
1,N/A,N/A,N/A
2,N/A,N/A,N/A
3,N/A,N/A,N/A
4,N/A,N/A,N/A


In [ ]:
import requests
import pandas as pd
import time

def scrape_bauhaus_muet():
    # Using the Shopify JSON endpoint for the collection
    base_url = "https://www.bauhaus.com.hk/en/collections/muet/products.json"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json"
    }

    extracted_data = []
    page = 1
    limit = 250

    print("Initiating scrape of the MUET collection...")

    while True:
        url = f"{base_url}?limit={limit}&page={page}"
        print(f"Requesting page {page}...")

        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                break

            data = response.json()
            products = data.get("products", [])
            if not products:
                break

            for product in products:
                variants = product.get("variants", [])
                images = product.get("images", [])

                extracted_data.append({
                    "Product Name": product.get("title"),
                    "Price": variants[0].get("price") if variants else "N/A",
                    "Image URL": images[0].get("src") if images else "N/A",
                    "Product URL": f"https://www.bauhaus.com.hk/en/products/{product.get('handle')}"
                })

            time.sleep(1)
            page += 1
        except Exception as e:
            print(f"Error on page {page}: {e}")
            break

    if extracted_data:
        df = pd.DataFrame(extracted_data)
        df.to_csv("bauhaus_muet_products.csv", index=False)
        print(f"\nScraping complete. Extracted {len(df)} products saved to 'bauhaus_muet_products.csv'.")
        display(df.head())
        return df
    else:
        print("No data retrieved.")
        return None

if __name__ == "__main__":
    df_scraped_products = scrape_bauhaus_muet()

Initiating scrape of the MUET collection...
Requesting page 1...
Requesting page 2...
Requesting page 3...

Scraping complete. Extracted 256 products saved to 'bauhaus_muet_products.csv'.


,Product Name,Price,Image URL,Product URL
0,MUET Noble Tote Bag,495.00,https://cdn.shopify.com/s/files/1/0715/6691/79...,https://www.bauhaus.com.hk/en/products/801225g...
1,MUET Diamond Three-layer Quilted Bag,445.00,https://cdn.shopify.com/s/files/1/0715/6691/79...,https://www.bauhaus.com.hk/en/products/801225g...
2,MUET Diamond Double-layer Quilted Bag,295.00,https://cdn.shopify.com/s/files/1/0715/6691/79...,https://www.bauhaus.com.hk/en/products/801225g...
3,MUET Diamond Purses Crossbody Bag,355.00,https://cdn.shopify.com/s/files/1/0715/6691/79...,https://www.bauhaus.com.hk/en/products/8012252...
4,MUET Stars 2WAY Quilted Bag,295.00,https://cdn.shopify.com/s/files/1/0715/6691/79...,https://www.bauhaus.com.hk/en/products/801225g...


## Integrating Scraped Data into `products_db`

This section processes the `df_scraped_products` DataFrame (created by the scraping script) and transforms it into the format required for the `products_db` dictionary. This allows the template filling function to use the actual product data.

In [ ]:
import pandas as pd
import os

# Configuration Defaults (Ensure these match your main config cell)
DEFAULT_PRICE = 89.90
DEFAULT_WEIGHT = 300
DEFAULT_LENGTH = 25
DEFAULT_WIDTH = 20
DEFAULT_HEIGHT = 10

try:
    # Load the scraped data from the CSV generated in the previous step
    if os.path.exists('bauhaus_muet_products.csv'):
        df_scraped_products = pd.read_csv('bauhaus_muet_products.csv')
        print(f"Loaded {len(df_scraped_products)} products from CSV.")

    if 'df_scraped_products' in globals() and not df_scraped_products.empty:
        new_products_db = {}

        for index, row in df_scraped_products.iterrows():
            product_name = str(row.get('Product Name', 'Unknown Product'))

            # Clean Price: Remove currency symbols and commas
            price_raw = str(row.get('Price', '0')).replace('HK$', '').replace(',', '').strip()
            try:
                price = float(price_raw) if price_raw not in ['N/A', ''] else DEFAULT_PRICE
            except:
                price = DEFAULT_PRICE

            image_url = row.get('Image URL', '')
            product_url = row.get('Product URL', '')

            # Map to the format required by the TikTok template filler
            new_products_db[product_name] = {
                "category": "Uncategorized",
                "description": f"Official MUET product: {product_name}. Imported from Bauhaus Hong Kong.",
                "leather_type": "Synthetic / Unknown",
                "main_image": image_url,
                "images": [image_url] if image_url else [],
                "color": "Multi",
                "size": "One Size",
                "price": price,
                "weight": DEFAULT_WEIGHT,
                "dimensions": (DEFAULT_LENGTH, DEFAULT_WIDTH, DEFAULT_HEIGHT),
                "product_url": product_url
            }

        # Update the global products_db variable
        products_db = new_products_db
        # Reset the upload tracker for this fresh batch
        uploaded = []

        print(f"Successfully updated products_db. Ready to generate upload file.")
        display(pd.DataFrame.from_dict(products_db, orient='index').head())
    else:
        print("Error: df_scraped_products is empty or CSV not found.")

except Exception as e:
    print(f"An error occurred during integration: {e}")

Loaded 256 products from CSV.
Successfully updated products_db. Ready to generate upload file.


,category,description,leather_type,main_image,images,color,size,price,weight,dimensions,product_url
MUET Noble Tote Bag,Uncategorized,Official MUET product: MUET Noble Tote Bag. Im...,Synthetic / Unknown,https://cdn.shopify.com/s/files/1/0715/6691/79...,[https://cdn.shopify.com/s/files/1/0715/6691/7...,Multi,One Size,495.0,300,"(25, 20, 10)",https://www.bauhaus.com.hk/en/products/801225g...
MUET Diamond Three-layer Quilted Bag,Uncategorized,Official MUET product: MUET Diamond Three-laye...,Synthetic / Unknown,https://cdn.shopify.com/s/files/1/0715/6691/79...,[https://cdn.shopify.com/s/files/1/0715/6691/7...,Multi,One Size,445.0,300,"(25, 20, 10)",https://www.bauhaus.com.hk/en/products/801225g...
MUET Diamond Double-layer Quilted Bag,Uncategorized,Official MUET product: MUET Diamond Double-lay...,Synthetic / Unknown,https://cdn.shopify.com/s/files/1/0715/6691/79...,[https://cdn.shopify.com/s/files/1/0715/6691/7...,Multi,One Size,295.0,300,"(25, 20, 10)",https://www.bauhaus.com.hk/en/products/801225g...
MUET Diamond Purses Crossbody Bag,Uncategorized,Official MUET product: MUET Diamond Purses Cro...,Synthetic / Unknown,https://cdn.shopify.com/s/files/1/0715/6691/79...,[https://cdn.shopify.com/s/files/1/0715/6691/7...,Multi,One Size,355.0,300,"(25, 20, 10)",https://www.bauhaus.com.hk/en/products/8012252...
MUET Stars 2WAY Quilted Bag,Uncategorized,Official MUET product: MUET Stars 2WAY Quilted...,Synthetic / Unknown,https://cdn.shopify.com/s/files/1/0715/6691/79...,[https://cdn.shopify.com/s/files/1/0715/6691/7...,Multi,One Size,295.0,300,"(25, 20, 10)",https://www.bauhaus.com.hk/en/products/801225g...


### Important Note on Web Scraping

Web scraping relies heavily on the structure (HTML/CSS) of the target website. The selectors used in the code (`product-item-info`, `product-item-link`, `price`, `product-image-photo`) are educated guesses based on common e-commerce site layouts. If the code doesn't extract the data correctly, you will need to:

1.  **Open the URL in a web browser.**
2.  **Right-click on a product element (e.g., name, price, image) and select "Inspect" or "Inspect Element".**
3.  **Examine the HTML structure** to find the correct `tag` and `class` or `id` attributes for the elements you want to extract.
4.  **Update the selectors** in the Python code accordingly.

For instance, if product names are inside an `<h2>` tag with a class `product-title`, you would change `container.find('a', class_='product-item-link')` to `container.find('h2', class_='product-title')`.

After successfully scraping the data, you can integrate it into your `products_db` dictionary for further processing.

In [ ]:
### Redundant Cell Removed
*Logic consolidated into the 'Integrating Scraped Data' section below.*

FileNotFoundError: [Errno 2] No such file or directory: 'bauhaus_muet_products.csv'